# CG Map-Making Demo

Wiener filter sky reconstruction at 25 MHz using `lusee.mapmaker`.  
Reproduces the approach of [Camacho et al. 2026](https://arxiv.org/abs/2508.16773).

Requires `LUSEE_DRIVE_DIR` pointing to the LuSEE-Night data (beam + sky FITS files).

In [ ]:
import os
os.environ["JAX_ENABLE_X64"] = "1"

import time
import jax
import jax.numpy as jnp
import numpy as np
import lusee
import healpy as hp
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
DRIVE = os.environ.get("LUSEE_DRIVE_DIR", "/fs/zack/LuSEE-Night/")
BEAM_FILE = DRIVE + "Simulations/BeamModels/LanderRegolithComparison/eight_layer_regolith/hfss_lbl_3m_75deg.fits"
SKY_FILE = DRIVE + "Simulations/SkyModels/ULSA_32_ddi_smooth.fits"

LMAX = 32
FREQ = np.array([25.0])  # single frequency
OBS_RANGE = "2025-02-01 13:00:00 to 2025-02-28 13:00:00"  # full sidereal rotation

## Build instrument and simulate data

In [ ]:
sim, beams, obs = lusee.mapmaker.build_instrument(
    beam_file=BEAM_FILE, obs_range=OBS_RANGE, freq=FREQ, lmax=LMAX,
)

# Load ULSA sky at 25 MHz
sky_full = lusee.sky.FitsSky(SKY_FILE, lmax=LMAX)
fi = int(np.argmin(np.abs(np.asarray(sky_full.freq) - 25.0)))
sky = lusee.sky.HealpixSky(
    sky_full.Nside, LMAX,
    maps=[hp.alm2map(np.asarray(sky_full.mapalm[fi]), sky_full.Nside, verbose=False)],
    freq=FREQ, frame="galactic",
)

# Simulate noiseless data + add radiometric noise
data_clean = sim.simulate(sky=sky)
sigma = 1.0  # K
data = data_clean + sigma * jax.random.normal(jax.random.PRNGKey(42), data_clean.shape)

print(f"{len(obs.times)} timesteps, {data.shape[1]} channels, SNR ~ {float(jnp.std(data_clean))/sigma:.0f}")

## Wiener filter solve via CG

`lusee.mapmaker.solve` uses `conj(jax.vjp)` for the Hermitian adjoint and `jax.scipy.sparse.linalg.cg` for the solve. See `docs/wirtinger_cg.md`.

In [ ]:
S_inv = lusee.mapmaker.compute_cl_prior(sky, LMAX)

t0 = time.time()
sky_hat = lusee.mapmaker.solve(sim, data, sky, sigma, signal_prior=S_inv, maxiter=50, tol=1e-8)
print(f"Solved in {time.time() - t0:.0f}s")

## Sky maps: input vs recovered

In [ ]:
true_alm = np.asarray(sky.mapalm[0])
rec_alm = np.asarray(sky_hat[0])
true_map = hp.alm2map(true_alm, sky.Nside, verbose=False)
rec_map = hp.alm2map(rec_alm, sky.Nside, verbose=False)
resid = rec_map - true_map

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, m, title, cmap in zip(axes,
        [true_map, rec_map, resid],
        ["ULSA input (25 MHz)", "CG reconstruction", "Residual"],
        ["inferno", "inferno", "RdBu_r"]):
    hp.mollview(m, title=title, hold=True, sub=(1, 3, list(axes).index(ax) + 1),
                cmap=cmap)
plt.tight_layout()

## Reconstruction fidelity: $\rho_\ell$ vs $\ell$

Cross-correlation coefficient between true and recovered sky per multipole (cf. Camacho+ 2026 Fig 5).

In [ ]:
cl_true = hp.alm2cl(true_alm)
cl_rec = hp.alm2cl(rec_alm)
cl_cross = hp.alm2cl(true_alm, rec_alm)
rho = cl_cross / np.sqrt(cl_true * cl_rec + 1e-30)
ell = np.arange(len(rho))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# rho vs ell
ax1.plot(ell[1:LMAX+1], rho[1:LMAX+1], 'o-', ms=4)
ax1.axhline(0, color='k', lw=0.5)
ax1.axhline(1, color='k', lw=0.5, ls='--')
ax1.set_xlabel(r"Multipole $\ell$")
ax1.set_ylabel(r"$\rho_\ell$")
ax1.set_title(f"Reconstruction fidelity at 25 MHz (mean $\\rho_{{1..10}}$ = {np.nanmean(rho[1:11]):.2f})")
ax1.set_ylim(-0.2, 1.1)
ax1.set_xlim(0, LMAX + 1)

# Power spectra
ax2.semilogy(ell[1:LMAX+1], cl_true[1:LMAX+1], 'k-', label="Input $C_\\ell$", lw=2)
ax2.semilogy(ell[1:LMAX+1], cl_rec[1:LMAX+1], 'r--', label="Recovered $C_\\ell$", lw=2)
ax2.set_xlabel(r"Multipole $\ell$")
ax2.set_ylabel(r"$C_\ell$")
ax2.set_title("Angular power spectrum")
ax2.legend()

plt.tight_layout()